# Hierarchy Expansion Benchmarking

This notebook demonstrates how to benchmark SNOMED CT hierarchy expansion methods using synthetic datasets.

## Overview

Hierarchy expansion evaluates how well a method can find related concepts through parent-child relationships in the SNOMED CT ontology.

### Key Metrics:
- **Exact Match Rate**: Fraction of cases where all expected concepts are found exactly
- **Recall@K**: Fraction of expected concepts found in top-K results
- **Precision@K**: Fraction of top-K results that are relevant
- **F1@K**: Harmonic mean of precision and recall at K
- **Jaccard Similarity**: Overlap between predicted and expected sets

### Data Format:
```python
{
    'seed_cui': str,              # Starting concept
    'expected_related': List[str]  # Expected related CUIs
}
```

In [ ]:
# Import required modules
from snomed_methods.benchmarking.hierarchy import (
    evaluate_hierarchy_expansion,
    generate_hierarchy_dataset,
)

## Generate Synthetic Hierarchy Dataset

In [ ]:
# Generate hierarchy expansion dataset
dataset = generate_hierarchy_dataset(num_samples=50)

print(f"Dataset size: {len(dataset)}")
print("\nFirst sample:")
sample = dataset[0]
print(f"  Seed CUI: {sample['seed_cui']}")
print(f"  Expected related ({sample['num_expected']} concepts):")
for cui in sample["expected_related"][:5]:
    print(f"    - {cui}")

## Create Mock Expansion Function

For demonstration, we create a mock expansion function. Replace with actual hierarchy traversal.

In [ ]:
# Example: Mock hierarchy expansion based on CUI hash patterns
def simple_expansion(seed_cui: str) -> list:
    """Mock hierarchy expansion returning synthetic related concepts."""

    # Generate some related CUIs deterministically from seed
    seed_hash = hash(seed_cui)

    related = [
        str(abs(seed_hash + i) % 1000000000).zfill(9)
        for i in range(2, 22)  # Generate 20 related concepts
    ]

    return related[:20]  # Return top 20 results


# Test the mock function
test_cui = dataset[0]["seed_cui"]
result = simple_expansion(test_cui)
print(f"Seed: {test_cui}")
print(f"Expanded concepts ({len(result)}):")
for cui in result[:5]:
    print(f"  - {cui}")

## Evaluate Hierarchy Expansion

Assess performance using different recall thresholds (K values).

In [ ]:
# Evaluate hierarchy expansion
results = evaluate_hierarchy_expansion(
    expansion_func=simple_expansion,
    dataset=dataset,
    k_values=[5, 10, 20],
)

print("\n=== Hierarchy Expansion Benchmarking Results ===")
for metric, value in results.items():
    if metric != "num_samples":
        print(f"{metric}: {value:.4f}")

print(f"\nTotal samples evaluated: {results['num_samples']}")

## Load Pre-generated Datasets

In [ ]:
from snomed_methods.benchmarking.hierarchy import load_hierarchy_datasets

# Load all pre-generated datasets
datasets = load_hierarchy_datasets()

for name, data in datasets.items():
    print(f"{name}: {len(data)} samples")

## Working with Real SNOMED Data

When actual SNOMED CT data is available, use the semantic expansion functionality:

In [ ]:
# Example: Using real SNOMED hierarchy expansion (requires UK data)
# from snomed_methods import SemanticSearch

# searcher = SemanticSearch(uk_path="/path/to/uk_sct2cl_42.2.0")

# def real_expansion(seed_cui):
#     # For demo, use a known concept from the dataset
#     result = searcher.search("concept", max_concepts=20)
#     return list(result.cuis)

# results_real = evaluate_hierarchy_expansion(real_expansion, dataset[:10])
# print(results_real)

## K Value Analysis

In [ ]:
# Compare different K values
k_options = [5, 10, 15, 20]

results_k = evaluate_hierarchy_expansion(
    expansion_func=simple_expansion,
    dataset=dataset[:30],
    k_values=k_options,
)

print("\nPerformance at different K values:")
print(f"{'K':<5} {'Recall':<12} {'Precision':<12} {'F1':<10}")
print("-" * 45)

for k in k_options:
    r = results_k.get(f"recall@{k}", 0)
    p = results_k.get(f"precision@{k}", 0)
    f1 = results_k.get(f"f1@{k}", 0)
    print(f"{k:<5} {r:<12.4f} {p:<12.4f} {f1:<10.4f}")

## Summary

This notebook demonstrated:
1. Generating synthetic hierarchy expansion datasets
2. Evaluating expand functions using multiple metrics
3. Working with pre-generated datasets for reproducibility
4. Analyzing performance at different K values